# MFLI and UHFLI hardware validation

Validation on 5 October 2026 used MFLI dev7920 and UHFLI dev20046, both reporting
an empty installed-options string. The environment was QCoDeS 0.56.0,
zhinst-qcodes 0.6.0, zhinst-toolkit 1.4.0, zhinst-core 26.7.2.5 and LabOne 26.07.
The instruments had no external signal or auxiliary connections.

The checks below passed. Physical signal outputs remained off. Temporary
settings were captured and restored; final comparisons against the original
snapshots matched all 217 MFLI settings and all 267 UHFLI settings exactly.
Connections made by the tests were disconnected afterwards.

| Check | MFLI | UHFLI |
| --- | --- | --- |
| Inherited node coverage and settings reads | 432 nodes mapped; 217 setting parameters read | 484 nodes mapped; 267 setting parameters read |
| Measurement channels and disabled-channel errors | Demodulator 0; demodulator 1 correctly rejected as reference-only | All eight demodulators |
| Output-amplitude aliases | Mixer 1 read/write verified with signal output off | Mixers 3 and 7 read/write verified with signal outputs off |
| Slow-stream timeout | Timeout raised without returning stale data | Timeout raised without returning stale data |
| Base scope acquisition | One channel, 16,384 samples | One channel, 16,384 samples |
| Scope FFT | 8,193 finite bins | 8,193 finite bins |
| DAQ module | X and Y, two bursts each, 32 samples per burst | X and Y, two bursts each, 32 samples per burst |
| Sweeper module | Three amplitude setpoints with signal output off | Three amplitude setpoints with signal output off |
| Input routing, metadata and demodulator controls | Voltage/current/auxiliary input selection; scaling guard; filter, harmonic, phase, sinc and oscillator settings; external-reference mode | Both voltage and auxiliary input selections; scaling guard; filter, harmonic, phase, sinc and oscillator settings; external-reference errors |
| Internal auxiliary-signal test | Approximately doubled R when the requested level doubled; a +90 degree reference-phase change produced a -90 degree reading change | Doubled R when the requested level doubled; a +90 degree reference-phase change produced a -90 degree reading change |
| Device-settings module | Saved and parsed a settings XML file | Saved and parsed a settings XML file |

The internal-signal test routed auxiliary output 0 back into demodulator 0
internally, using nominal 10 mV and 20 mV DC levels and a zero-frequency
oscillator. It checked relative amplitude and phase response. It does not
calibrate the physical signal-input path or establish the RMS response to an
external sinusoid. Device quantization affected the nominal auxiliary levels:
the measured amplitude ratios were 2.03447 on the MFLI and 2.00000 on the UHFLI.

Two QCoDeS datasets measured all nine available measurement demodulators together,
once without threads and once with threads. Each contained 38 result columns
at three setpoints: four quantities per demodulator, plus a separate MFLI R
parameter and the legacy complex parameter. Saved units and the X/Y-to-R/Theta
relationships were checked. All 51 hardware-free lock-in and existing Zurich
HDAWG regression tests also passed. These checks required no changes to the
driver implementation from step 3.

## Measurement notebook verification

All 14 code cells of the [measurement example](ZurichInstruments_lockins.ipynb)
were executed on each instrument with signal outputs off. Each run acquired
individual X/Y/R/Theta points, a coherent four-component point, a manual X/Y
time trace, and the following QCoDeS datasets:

| Example | Saved data on each instrument |
| --- | --- |
| Point `do1d` | X/Y/R/Theta at 11 frequencies |
| Point `do2d` | X/Y/R/Theta at 3 amplitudes by 11 frequencies |
| Trace `do0d` | X/Y at 201 time setpoints |
| Trace `do1d` | X/Y at 5 frequencies by 201 time setpoints |

The saved array shapes, voltage/time units and actual device timestamp axes
were checked. The validation harness restored the starting instrument settings
and connections after running the notebook's cleanup cell. The example itself
turns off excitation and streaming; it does not restore an earlier configuration.

The trace helper also has hardware-free tests for timestamp conversion on both
clock rates, incomplete or misaligned bursts, non-finite values, timeout cleanup,
configuration changes, and `do0d`/`do1d` dataset coordinates. All **65** notebook,
lock-in and existing Zurich HDAWG tests passed. The notebooks have cleared outputs
and disable automatic execution during documentation builds, as these examples
require connected instruments.

## Observed device behaviour

Requested settings may be quantized by the hardware. For example, a requested
2 ms time constant read back as approximately 1.973 ms on the MFLI and 2.071 ms
on the UHFLI. Read back the accepted setting when the exact value matters.

The acquisition timeout is not a hard wall-clock deadline. With a 40 ms local
timeout and deliberately slow streams, the timeout errors arrived after about
0.52 s and 0.59 s respectively because an individual LabOne sample call had to
return first. The default five-second timeout was sufficient for the other
checks.

DAQ timestamp conversion requires the device's clock rate. Pass
`clk_rate=instrument.clockbase()` to the inherited DAQ module's `read`
method, especially on the UHFLI; the vendor method defaults to 60 MHz, while the
tested UHFLI reports 1.8 GHz. See the
[vendor DAQ example](https://docs.zhinst.com/zhinst-toolkit/en/latest/examples/daq_module.html).

The scope-module examples in older documentation set `averager.weight`.
LabOne 26.07 rejects that setting with the default uniform averaging method.
The validation used the module's default averaging configuration for single
records. To set an exponential weight, first select the corresponding averaging
method.

Restoration exposed one MFLI quirk: auxiliary outputs 2 and 3 initially selected
demodulator 1, although this base instrument cannot select that measurement
channel through an ordinary write. Changing their output sources reset those
selections to zero. Reloading the built-in parameter preset recovered the two
original selections, and all 217 original setting values then matched exactly.
No preset was saved or erased. The final internal-signal test touched only
auxiliary output 0 and restored successfully without reloading a preset.
A settings snapshot alone is therefore not a guarantee that every factory
default can be replayed through individual writes.

A separate reproduction using only `zhinst.core` narrowed this down further:
with `auxouts/2/outputselect=0` and the factory `demodselect=1`, reading
`demodselect` and writing that same value back changed it to zero without an
exception. Reloading the parameter preset restored one. This behaviour occurs
below the new QCoDeS wrappers; whether its cause is firmware, data-server
validation or the factory preset remains undetermined.

## Remaining cable checks

The next check needs a physical signal-output-to-input loopback cable. It should
verify sinusoidal RMS amplitude, physical input scaling, phase response, and
acquisition while sweeping a real signal. External-reference locking and
physical trigger behaviour also need an appropriate signal connection.

The successful external-reference checks above verify mode selection and
readout rejection on reserved channels; they do not demonstrate lock acquisition.
Physical output drive, input gain accuracy, external triggering and maximum
scope memory length have not been validated by this run. Licensed options were
not exercised.


[Return to the measurement example](ZurichInstruments_lockins.ipynb).